# 04_Submission

## 1. Project Setup

### 1.1: Imports and Path

In [1]:
from pathlib import Path
import sys
import torch
import pandas as pd

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TEST_DIR = DATA_DIR / "test_images"
RESULTS_DIR = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Test directory: {TEST_DIR}")
print(f"Test directory exists: {TEST_DIR.exists()}")

Project root: /Users/amitkumar/Desktop/Computing/Projects/paddy-disease-classification
Test directory: /Users/amitkumar/Desktop/Computing/Projects/paddy-disease-classification/data/raw/test_images
Test directory exists: True


### 1.2: Set Device

In [7]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: mps


## 2. Inspect the test data

In [2]:
test_image_paths = sorted(TEST_DIR.glob("*.jpg"))

print(f"Number of test images: {len(test_image_paths)}")
print("\nFirst 10 image IDs:")
for path in test_image_paths[:10]:
    print(path.name)

Number of test images: 3469

First 10 image IDs:
200001.jpg
200002.jpg
200003.jpg
200004.jpg
200005.jpg
200006.jpg
200007.jpg
200008.jpg
200009.jpg
200010.jpg


## 3. Define The Test Dataset

In [3]:
from PIL import Image
from torch.utils.data import Dataset


class TestImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, image_path.name

## 4. Test Preprocessing

In [4]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print(test_transform)

Compose(
    Resize(size=384, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(384, 384))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 5. Create the test dataloader

In [5]:
from torch.utils.data import DataLoader

test_dataset = TestImageDataset(
    image_paths=test_image_paths,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of test batches: {len(test_loader)}")

Number of test samples: 3469
Number of test batches: 109


## 6. Load The Trained Model

In [9]:
from torch import nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

from src.data import CLASS_NAMES

NUM_CLASSES = len(CLASS_NAMES)

weights = EfficientNet_V2_S_Weights.DEFAULT

model = efficientnet_v2_s(weights=weights)

model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

# Recreate the Stage-6 fine-tuning configuration
for parameter in model.features.parameters():
    parameter.requires_grad = False

for parameter in model.features[6:].parameters():
    parameter.requires_grad = True

model = model.to(device)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")

Training samples: 8325
Validation samples: 2082
Number of classes: 10
Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']


load the saved checkpoint:

In [10]:
checkpoint_path = (
    PROJECT_ROOT
    / "models"
    / "efficientnet_v2_s_finetuned_stage6_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])

print(f"Loaded checkpoint from epoch: {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

Loaded checkpoint from epoch: 12
Best validation loss: 0.1959


## 7. Sanity Check The Model

In [11]:
images, image_ids = next(iter(test_loader))

print(f"Image batch shape: {images.shape}")
print(f"Image dtype: {images.dtype}")
print(f"Number of image IDs: {len(image_ids)}")

with torch.inference_mode():
    images = images.to(device)
    logits = model(images)

print(f"Logits shape: {logits.shape}")

Image batch shape: torch.Size([32, 3, 384, 384])
Image dtype: torch.float32
Number of image IDs: 32
Logits shape: torch.Size([32, 10])


## 8. Generate Prediction

In [12]:
model.eval()

all_predictions = []
all_image_ids = []

with torch.inference_mode():
    for images, image_ids in test_loader:
        images = images.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_image_ids.extend(image_ids)

print(f"Number of predictions: {len(all_predictions)}")
print(f"Number of image IDs: {len(all_image_ids)}")

Number of predictions: 3469
Number of image IDs: 3469


Convert class image to class names:

In [13]:
predicted_labels = [
    CLASS_NAMES[prediction]
    for prediction in all_predictions
]

print("First 10 predictions:")
for image_id, label in zip(all_image_ids[:10], predicted_labels[:10]):
    print(f"{image_id}: {label}")

First 10 predictions:
200001.jpg: hispa
200002.jpg: normal
200003.jpg: blast
200004.jpg: blast
200005.jpg: blast
200006.jpg: brown_spot
200007.jpg: dead_heart
200008.jpg: brown_spot
200009.jpg: hispa
200010.jpg: normal


## 9. Create Data Frame

In [14]:
submission = pd.DataFrame({
    "image_id": all_image_ids,
    "label": predicted_labels
})

submission.head(10)

,image_id,label
0,200001.jpg,hispa
1,200002.jpg,normal
2,200003.jpg,blast
3,200004.jpg,blast
4,200005.jpg,blast
5,200006.jpg,brown_spot
6,200007.jpg,dead_heart
7,200008.jpg,brown_spot
8,200009.jpg,hispa
9,200010.jpg,normal


## 10. Validate The Submission
This is important because a correctly formatted CSV with a missing or duplicated image ID would still be a bad submission.

In [15]:
assert len(submission) == len(test_image_paths), (
    "Number of submission rows does not match the number of test images."
)

assert submission["image_id"].is_unique, (
    "Duplicate image IDs found in the submission."
)

assert set(submission["image_id"]) == {
    path.name for path in test_image_paths
}, (
    "Submission image IDs do not match the test images."
)

assert submission["label"].isin(CLASS_NAMES).all(), (
    "Submission contains an invalid class label."
)

print("Submission validation passed.")
print(f"Rows: {len(submission)}")
print(f"Columns: {list(submission.columns)}")
print(f"Unique image IDs: {submission['image_id'].nunique()}")
print(f"Unique labels: {submission['label'].nunique()}")

Submission validation passed.
Rows: 3469
Columns: ['image_id', 'label']
Unique image IDs: 3469
Unique labels: 10


## 11. Save the submission

In [16]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

submission_path = RESULTS_DIR / "submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print(f"Submission saved to: {submission_path}")

Submission saved to: /Users/amitkumar/Desktop/Computing/Projects/paddy-disease-classification/results/submission.csv


Inspect the csv file:

In [17]:
final_submission = pd.read_csv(submission_path)

print(final_submission.head())
print()
print(f"Shape: {final_submission.shape}")
print(f"Columns: {list(final_submission.columns)}")
print(f"Missing values: {final_submission.isna().sum().sum()}")

     image_id   label
0  200001.jpg   hispa
1  200002.jpg  normal
2  200003.jpg   blast
3  200004.jpg   blast
4  200005.jpg   blast

Shape: (3469, 2)
Columns: ['image_id', 'label']
Missing values: 0
